# Análise das Transcrições das Lives do Frei Gilson - Quaresma 2025

## Introdução

**Autor:** *Bruno Conterato*

**Objetivo:** *Analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025 utilizando técnicas modernas de processamento de linguagem natural (NLP).*

---

## Descrição
Este notebook tem como objetivo analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025.

---

## Etapas de Pré-processamento
Antes da análise, o notebook prepara as transcrições para que o conteúdo fique mais organizado e fácil de processar.

### O que é feito
1. **Limpeza básica do texto**
   - remove espaços extras no início e no fim de cada linha;
   - padroniza a formatação da transcrição.

2. **Divisão em blocos menores**
   - separa a transcrição em chunks de tamanho controlado;
   - mantém uma sobreposição entre blocos para preservar contexto.

3. **Identificação de referências bíblicas**
   - detecta quando há citação explícita de um livro, capítulo e versículo;
   - extrai essas referências em formato estruturado.

4. **Filtro de conteúdo relevante**
   - orienta o modelo a considerar apenas o que foi dito durante o Rosário;
   - ignora a reflexão final e outros trechos que não fazem parte da análise principal.

---

## Necessidades de Processamento
Para analisar corretamente as transcrições, o notebook precisa lidar com alguns tipos de conteúdo diferentes:

1. **Separar os momentos da fala**
   - identificar o ponto de divisão entre as reflexões do Terço e as reflexões do Dia;
   - separar os textos em duas partes: reflexão do Terço e reflexão do Dia.

2. **Filtrar trechos que não entram na análise principal**
   - remover as seções marcadas com a tag `[Música]`;
   - identificar e remover orações oficiais da Igreja Católica.

3. **Registrar músicas citadas durante a live**
   - extrair e documentar as músicas cantadas;
   - registrar o nome da música e o autor de cada uma.

4. **Contextualizar os ensinamentos dentro do Rosário**
   - considerar em que momento do Rosário cada ensinamento foi apresentado;
   - relacionar cada trecho ao terço e ao mistério meditado naquele instante.

---

## Recursos Externos Utilizados
Além do texto do notebook, este processo depende de alguns recursos que ficam fora dele, mesmo quando estão no mesmo projeto local:

1. **Vector Store da Bíblia**
   - armazenado em `../Bíblia VectorStore/biblia_vectorstore`;
   - usado para buscar passagens bíblicas semelhantes aos trechos analisados.

2. **Banco SQLite da Bíblia**
   - acessado em `../Bíblia VectorStore/biblia.db`;
   - contém os versículos consultados durante a extração das referências.

3. **Modelo de embeddings**
   - `sentence-transformers/all-MiniLM-L6-v2`;
   - transforma trechos de texto em vetores para permitir a busca por similaridade.

4. **Modelo de linguagem local**
   - executado via `Ollama`;
   - gera as respostas e faz a extração estruturada das informações.

5. **Arquivos de entrada e saída do projeto**
   - leitura em `../../data/raw/Santo Rosário | Quaresma 2025/Youtube to Text`;
   - saída prevista em `../../data/processed/Santo Rosário | Quaresma 2025/Youtube to Text`.


## 1. Configurações

### 1.1. Importação de Bibliotecas


In [1]:
import os
from typing import List, Optional


import pandas as pd
import sqlite3
from tqdm.notebook import tqdm
from pprint import pprint

from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.output_parsers.enum import EnumOutputParser, Enum
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers.pydantic import PydanticOutputParser
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field, field_validator
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain.output_parsers import RetryOutputParser
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model
from langchain.output_parsers import OutputFixingParser
from langchain_core.exceptions import OutputParserException

from utils import BookEnum, ORDERED_BOOKS, BinaryResponse

from dotenv import load_dotenv

load_dotenv()

True

In [ ]:

# MODEL = "Gemma-3-Gaia-PT-BR-4b-it-f16:latest"
MODEL = "batiai/gemma4-e4b:q4"

# List of Models: https://ai.google.dev/gemini-api/docs/models
# MODEL = "gemini-2.0-flash"

# Providers: azure_openai, deepseek, bedrock, openai, fireworks, xai, mistralai,
# ollama, google_anthropic_vertex, cohere, google_vertexai, perplexity,
# azure_ai, google_genai, ibm, bedrock_converse, groq, together, anthropic, huggingface
MODEL_PROVIDER = "ollama"
# MODEL_PROVIDER = "google_genai"

### 1.2. Hiperparâmetros

**Chunk size / Overlap**

| Métrica | Valor Associado |
| :--- | :--- |
| **Percentil p0** | 2 |
| **Percentil p10** | 32 |
| **Percentil p25 (Q1)** | 65 |
| **Percentil p50 (Mediana)** | 94 |
| **Percentil p75 (Q3)** | 135 |
| **Percentil p90** | 176 |
| **Percentil p95** | 202 |
| **Percentil p99** | 256 |
| **Percentil p100** | 576 |

*Nota: Os valores acima referem-se aos quantis (percentis) fornecidos. O número de caracteres por versículo bíblico não foi calculado com os dados apresentados, mas pode ser adicionado a esta tabela conforme o processamento das referências.*


In [ ]:
MIN_SIMILARITY_THRESHOLD = 0.65
CHUNK_SIZE = 800
CHUNK_OVERLAP = 200

## 2.0. Processamento de texto

In [4]:
def trim_line_whitespace(text):
    """
    Remove todos os tabs e espaços em branco do início e do final de cada linha do texto.
    """
    lines = text.split("\n")
    cleaned_lines = [line.strip() for line in lines]
    return "\n".join(cleaned_lines)

In [5]:
system_message = """
    Você é um especialista na fé católica.  
    Receberá um **texto com a transcrição da oração do Santo Rosário**, rezado ao vivo pelo Frei Gilson durante um dia da Quaresma de 2025.

    A oração está dividida em duas partes:
    - A primeira parte é o Santo Rosário, onde Frei Gilson reza o Rosário com os fiéis.
    - A segunda parte é uma reflexão relacionada à fé católica e ao silêncio.

    ⚠️ **IMPORTANTE:**  
    Ignore qualquer parte da transcrição que seja da parte da **reflexão**.  
    Use **apenas o que for dito durante a oração do Rosário**.

    ---

    ### Sua missão é extrair e organizar as informações nos itens de 1 a 5 abaixo:

    ---

    ## 1. Temática principal

    - Identifique a principal ideia que Frei Gilson ensina enquanto reza o Rosário.
    - Resuma em até 3 parágrafos.
    - Use somente ensinamentos que ele falou **durante a oração do Santo Rosário**.

    ---

    ## 2. Temáticas secundárias

    - Identifique de 2 a 5 temas que Frei Gilson também falou **durante a oração do Santo ROsário**.
    - Para cada tema:
    - Dê um título claro (ex: *Perseverança na fé*).
    - Escreva 1 a 2 parágrafos com os ensinamentos relacionados.

    ---

    ## 3. Versículos da Bíblia

    Vou lhe fornecer os **versículos bíblicos citados durante a oração**.
    Você só ignorará versículos que sejam referentes a orações como o Pai nosso, Ave Maria, Credo etc,
    mas manterá todos os demais versículos.
    Não se esqueça de nenhum versículo ou intervalo de versículos que eu lhe fornecer, a menos que citado antyeriormente nas excessões.
    Traga apenas versículos que foram citados durante a oração do Santo Rosário.
    Traga a localização do versículo, a transcrição integral do versículo e os ensinamentos que foram transmitidos por aquele versículo e pela narração do Frei Gilson.
    Para cada um:

    - Escreva assim:  
    `(Livro Capítulo, Versículo)`: <Transcrição integral do versículo> ou
    `(Livro Capítulo, Versículo-Início–Versículo-Fim)`: <Transcrição integral do intervalo de versículos>

    - Logo abaixo, escreva:
    **Ensinamentos:**: <Parágrafo ou frase que explique o que o Frei falou sobre esse versículo durante a oração>

    Se o versículo for relacionado a algum mistério do Santo Rosário (como `Segundo Mistério Doloroso` ou `Quinto Mistério Gozozo`), diga isso.

    ---

    ## 4. Músicas

    Se Frei Gilson mencionar músicas durante a oração:

    - Diga o nome da música e o artista, como neste exemplo:
    <Nome da música> - <Artista>: <contexto>  

    - Depois escreva o que Frei Gilson disse sobre a música.

    ---

    ## 5. Eventos de Agenda

    Se Frei Gilson mencionar **eventos** (missas, encontros, lives etc.):
    Traga todos os eventos citados durante a oração ou ao final do Santo Rosário.
    - Traga o nome do evento, a data e o local.
    - Depois escreva o que ele falou sobre esse evento.

    ---

    ## ⚠️ Regras finais:

    - NÃO copie orações conhecidas (Pai Nosso, Ave Maria, Credo etc.).
    - Não explique o rosário, exceto se dor necessário para compreender o respectivo ensinamento.
    - NÃO traga informações da reflexão final que ocorre após o rosário. Traga só o que é dito durante o Rosário.
    - NÃO invente nada. Só use o que estiver escrito.
    - NÃO deixe seções em branco. Se algo não aparecer, **remova a seção inteira**.
    ```
"""

## 3.0 Detecção de trechos da bíblia

In [ ]:
# 0. Carregue a transcrição
transcription = """
Hoje refletimos sobre a importância de sermos humildes. Como está escrito: "Bem-aventurados os humildes, pois herdarão a terra".
Mais adiante, mencionou-se que devemos amar o próximo como a nós mesmos.
"""

# 1. Divida a transcrição
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.create_documents([transcription])

# 2. Carregue os embeddings e a base vetorial
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
db = Chroma(
    collection_name="biblia",
    persist_directory="../Bíblia VectorStore/biblia_vectorstore",
    embedding_function=embedding_model,
)


# 3. Defina a enum e o parser
binary_parser = EnumOutputParser(enum=BinaryResponse)


# 4. Modelo Pydantic para saída estruturada de referência bíblica
class BibleReference(BaseModel):
    book: BookEnum = Field(
        ...,
        description="Nome do livro da Bíblia explicitamente anunciado. Não deve conter número do capítulo.",
        examples=ORDERED_BOOKS,
    )
    chapter: int = Field(
        ...,
        description="Número do capítulo explicitamente anunciado. Não deve conter número de versículos.",
    )
    verse_start: int = Field(
        ...,
        description="Número do primeiro versículo do intervalo explicitamente anunciado, se for um único versículo. Se for um intervalo, é o primeiro versículo do intervalo.",
    )
    verse_end: int = Field(
        ...,
        description="Número do último versículo do intervalo explicitamente anunciado, se for um intervalo. Se não houver intervalo, igual ao verse_start.",
    )

    @field_validator("verse_end", mode="before")
    @classmethod
    def set_verse_end(cls, v, values):
        if v is None:
            return values.get("verse_start")
        return v


# Parser para saída estruturada
bible_reference_parser = JsonOutputParser(pydantic_object=BibleReference)

# Novo prompt: pede saída JSON estruturada para referência bíblica
bible_reference_prompt_template = """
Você é um assistente que analisa trechos de transcrição de fala e identifica qual é a **passagem bíblica** citada de forma clara e explícita.

Sua tarefa é identificar e **extrair** as seguintes informações, mesmo que existam variações de linguagem falada ou erros de transcrição:
- O **nome do livro da Bíblia**
- O **número do capítulo**
- O **número do(s) versículo(s)** — pode ser um único versículo (ex: 3) ou um intervalo (ex: 3-5)

Sua resposta deve ser exatamente neste formato **sem explicações, comentários ou qualquer outro texto**:
{format_instructions}

Texto a ser analisado:
{query}
"""

bible_reference_prompt = PromptTemplate(
    template=bible_reference_prompt_template,
    input_variables=["query"],
    partial_variables={
        "format_instructions": bible_reference_parser.get_format_instructions()
    },
)

print("Format Instructions:")
pprint(bible_reference_parser.get_format_instructions())
# 5. LLM local
# Funciona bem com os modelos instruct:
# - llama3.1:8b-instruct-q5_K_M
# - mistral:7b-instruct
llm = ChatOllama(model=MODEL)

# 6. Cadeia completa manual (RAG customizado)


def format_context(docs: List[Document], show_option=False):
    if show_option:
        return "\n\n".join(
            f"Excerpt {i+1}: {doc.metadata["book"]} {doc.metadata["chapter"]}:{doc.metadata["verse"]} {doc.page_content}"
            for (i, doc) in enumerate(docs)
        )
    return "\n\n".join(
        f"{doc.metadata["book"]} {doc.metadata["chapter"]}:{doc.metadata["verse"]} {doc.page_content}"
        for doc in docs
    )


def retrieve_similar_bible_passages(
    query: str, k: int = 3, min_similarity_threshould: float = MIN_SIMILARITY_THRESHOLD
):
    """
    Recupera as passagens bíblicas mais similares ao trecho fornecido.
    """
    retriever = db.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": min_similarity_threshould},
    )
    return retriever.invoke(query)[:k]


def retrieve_books_and_chapters(docs: List[Document]) -> str:
    """
    Retrieve all verses from the specific book and chapter found in docs.
    Assumes a table 'versiculos' with columns: id, book, chapter, verse, text, line_number.
    """
    conn = sqlite3.connect("../Bíblia VectorStore/biblia.db")
    cursor = conn.cursor()
    results = []
    seen = set()
    for doc in docs:
        book = doc.metadata.get("book")
        chapter = doc.metadata.get("chapter")
        if book and chapter and (book, chapter) not in seen:
            seen.add((book, chapter))
            results.append(f"\n\nLivro: {book} - Capítulo: {chapter}")
            cursor.execute(
                "SELECT book, chapter, verse, text FROM versiculos WHERE book=? AND chapter=? ORDER BY verse ASC",
                (book, chapter),
            )
            for row in cursor.fetchall():
                results.append(f"{row[2]}: {row[3]}")
    conn.close()
    return "\n".join(results) if results else "Nenhum resultado encontrado."

README.md: 0.00B [00:00, ?B/s]

Format Instructions:
('The output should be formatted as a JSON instance that conforms to the JSON '
 'schema below.\n'
 '\n'
 'As an example, for the schema {"properties": {"foo": {"title": "Foo", '
 '"description": "a list of strings", "type": "array", "items": {"type": '
 '"string"}}}, "required": ["foo"]}\n'
 'the object {"foo": ["bar", "baz"]} is a well-formatted instance of the '
 'schema. The object {"properties": {"foo": ["bar", "baz"]}} is not '
 'well-formatted.\n'
 '\n'
 'Here is the output schema:\n'
 '```\n'
 '{"$defs": {"BookEnum": {"description": "_summary_\\n\\nArgs:\\n    Enum '
 '(_type_): _description_\\n\\nGot books list using '
 'query:\\n\\n```sql\\nSELECT DISTINCT book\\nFROM versiculos\\nORDER BY '
 'line_number\\n\\n```", "enum": ["Gênesis", "Êxodo", "Levítico", "Números", '
 '"Deuteronômio", "Josué", "Juízes", "Rute", "1 Samuel", "2 Samuel", "1 Reis", '
 '"2 Reis", "1 Crônicas", "2 Crônicas", "Esdras", "Neemias", "Tobias", '
 '"Judite", "Ester", "Jó", "Salmos"

In [7]:
# Pipeline binário para detecção de referência bíblica
binary_bible_reference_template = """
Você é um assistente que analisa trechos de transcrição de fala e identifica se há uma **citação bíblica clara e explícita**.

Sua tarefa é:
1. Verificar se o trecho contém uma **anunciação clara** de que será citada uma **passagem bíblica**.
2. A citação deve conter, de forma explícita (mesmo com erros de transcrição ou variações da fala):
   - O **nome do livro da Bíblia**
   - O **número do capítulo**
   - O **número do(s) versículo(s)** — pode ser um único versículo (ex: 3) ou um intervalo (ex: 3-5)

Se você encontrar essa citação bíblica clara, responda apenas com "Sim.". Caso contrário, responda apenas com "Não.".
Não escreva nenhuma explicação, justificativa ou comentário.
O resultado deve ser apenas "Sim." ou "Não." — nada além disso.

Texto a ser analisado:
{query}
"""

binary_bible_reference_prompt = PromptTemplate(
    template=binary_bible_reference_template,
    input_variables=["query"],
)

binary_bible_reference_chain = (
    {"query": lambda x: x["query"]}
    | binary_bible_reference_prompt
    | llm
    | binary_parser
)

In [8]:
fixing_parser = OutputFixingParser.from_llm(parser=bible_reference_parser, llm=llm)

bible_reference_chain = {"query": lambda x: x["query"]} | bible_reference_prompt | llm


def get_bible_passages(transcription: str, verbose: bool = False) -> list:
    """
    Extrai referências bíblicas de uma transcrição, com prints opcionais para depuração.
    """
    bible_passages = []
    chunks = splitter.create_documents([transcription])
    if verbose:
        print(f"[get_bible_passages] Total de chunks: {len(chunks)}")

    for idx, chunk in enumerate(tqdm(chunks, desc="Processando chunks")):
        found_reference = False
        if verbose:
            print(f"\n[get_bible_passages] Chunk {idx+1}/{len(chunks)}:")
            print(f"Conteúdo do chunk:\n{chunk.page_content}")

        # 1. Primeiro, verifique se há referência bíblica usando o pipeline binário
        try:
            binary_result = binary_bible_reference_chain.invoke(
                {"query": chunk.page_content}
            )
            if verbose:
                print(f"Resultado do pipeline binário: {binary_result}")
            if binary_result == BinaryResponse.YES:
                found_reference = True
        except Exception as e:
            if verbose:
                print(f"[Aviso] Erro no pipeline binário: {e}")
            continue

        if not found_reference:
            if verbose:
                print("Nenhuma referência bíblica clara detectada neste chunk.")
            continue

        # 2. Se sim, execute o pipeline estruturado (tenta parser normal, depois fixing_parser)
        try:
            output = bible_reference_chain.invoke({"query": chunk.page_content})
            print("Output:", output.content)
            try:
                ref = bible_reference_parser.parse(str(output.content))
            except OutputParserException:
                if verbose:
                    print("[Aviso] Parsing falhou, tentando OutputFixingParser...")
                ref = fixing_parser.parse(str(output.content))
            if verbose:
                print(f"Referência bíblica extraída: {ref}")
            bible_passages.append(ref)
        except Exception as e:
            if verbose:
                print(f"[Aviso] Erro ao extrair referência estruturada: {e}")
            pass

    if verbose:
        print(
            f"\n[get_bible_passages] Total de referências extraídas: {len(bible_passages)}"
        )
    # Alternativa simples: usar pandas para remover duplicatas de dicionários

    # Se os itens forem dicionários ou BaseModel, converta para dicionário
    dicts = [
        passage.dict() if hasattr(passage, "dict") else passage
        for passage in bible_passages
    ]
    unique_passages = pd.DataFrame(dicts).drop_duplicates().to_dict(orient="records")
    return unique_passages


# Exemplo de uso:
bible_refs = get_bible_passages(
    transcription=trim_line_whitespace(
        """
        Livro de Primeiro João, capítulo 3, versículos 16 a 17.
        
        Não sabeis que sois o Templo de Deus, e que o Espírito de Deus habita em vós?
        Se alguém destruir o Templo de Deus, Deus o destruirá. 
        Porque o templo de Deus é sagrado – e isso sois vós.
        """
    ),
    verbose=True,
)
print("Referências bíblicas extraídas:")
for ref in bible_refs:
    print(ref)

[get_bible_passages] Total de chunks: 1


Processando chunks:   0%|          | 0/1 [00:00<?, ?it/s]


[get_bible_passages] Chunk 1/1:
Conteúdo do chunk:
Livro de Primeiro João, capítulo 3, versículos 16 a 17.

Não sabeis que sois o Templo de Deus, e que o Espírito de Deus habita em vós?
Se alguém destruir o Templo de Deus, Deus o destruirá.
Porque o templo de Deus é sagrado – e isso sois vós.
Resultado do pipeline binário: BinaryResponse.YES
Output: ```json
{
  "book": "1 João",
  "chapter": 3,
  "verse_start": 16,
  "verse_end": 17
}
```
Referência bíblica extraída: {'book': '1 João', 'chapter': 3, 'verse_start': 16, 'verse_end': 17}

[get_bible_passages] Total de referências extraídas: 1
Referências bíblicas extraídas:
{'book': '1 João', 'chapter': 3, 'verse_start': 16, 'verse_end': 17}


## 4.0. Leitura dos Arquivos

In [9]:
model = init_chat_model(MODEL, model_provider=MODEL_PROVIDER)

raw_folder = "../../data/raw/Santo Rosário | Quaresma 2025/Youtube to Text"
processed_folder = "../../data/processed/Santo Rosário | Quaresma 2025/Youtube to Text"
titulo_template = (
    "Santo Rosário | Quaresma 2025 | 03:40 | {ordem}° Dia | Live Ao vivo.txt"
)

print("Diretório atual:", os.getcwd())


def gerar_titulo_fonte(ordem):
    return titulo_template.format(ordem=ordem)


for i in tqdm(range(13, 14), desc="Processando arquivos"):
    titulo_source = gerar_titulo_fonte(str(i))
    arquivo = f"{raw_folder}/{titulo_source}"
    titulo_md = f"{titulo_source[:-3]}.md"

    if os.path.exists(arquivo):
        with open(arquivo, "r+", encoding="utf-8") as f:
            conteudo = f.read()
            if conteudo:
                # TODO: retirar limites
                conteudo = conteudo[:20000]  # Limitar a 20.000 caracteres inicialmente
                print(f"\n\nArquivo: {titulo_source}")

                bible_refs = get_bible_passages(conteudo, verbose=True)
                print("Referências bíblicas extraídas:")
                for ref in bible_refs:
                    print(ref)

                messages = [
                    {"role": "system", "content": trim_line_whitespace(system_message)},
                    {"role": "user", "content": conteudo},
                ]

                chunks = []
                for chunk in model.stream(messages):
                    chunks.append(chunk.text())
                    print(chunk.text(), end="", flush=True)
                response = "".join(chunks)

                # with open(
                #     f"{processed_folder}/rosario_{titulo_md}", "w+", encoding="utf-8"
                # ) as f:
                #     f.write(response)

            else:
                print(f"\n\nO arquivo {titulo_source} está vazio.")
    else:
        print(f"\n\nArquivo não encontrado: {arquivo}")

    break

Diretório atual: /home/bruno/Workspace/MariaGPT/src/Rosários Quaresma Frei Gilson 2025


Processando arquivos:   0%|          | 0/1 [00:00<?, ?it/s]



Arquivo: Santo Rosário | Quaresma 2025 | 03:40 | 13° Dia | Live Ao vivo.txt
[get_bible_passages] Total de chunks: 39


Processando chunks:   0%|          | 0/39 [00:00<?, ?it/s]


[get_bible_passages] Chunk 1/39:
Conteúdo do chunk:
(144) Santo Rosário | Quaresma 2025 | 03:40 | 13° Dia | Live Ao vivo - YouTube
https://www.youtube.com/watch?v=5zDuvPECDoA
Resultado do pipeline binário: BinaryResponse.NO
Nenhuma referência bíblica clara detectada neste chunk.

[get_bible_passages] Chunk 2/39:
Conteúdo do chunk:
Transcript:
(00:05) [Música] [Música] [Música] [Música] [Música]
(01:16) [Música] [Música] [Música] [Música] [Música]
(02:45) [Música] [Música] [Música] [Música] [Música]
(04:14) [Música] [Música] [Música] [Música]
(05:25) [Música] [Música] [Música] [Música] [Música] [Música]
(07:02) [Música] [Música] [Música] [Música] [Música] [Música]
(08:36) [Música] [Música] [Música] [Música] [Música] C [Música]
(10:12) [Música] [Música] [Música] m [Música] [Música]
(11:41) [Música] [Música] [Música] J [Música]
(12:51) [Música] k [Música] [Música] [Música] AM [Música] [Música] [Música]
Resultado do pipeline binário: BinaryResponse.NO
Nenhuma referência bíblica clara dete